<a href="https://colab.research.google.com/github/donnabrown77/machine-learning-zoomcamp-homework/blob/main/homework_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:
!wget https://github.com/SVizor42/ML_Zoomcamp/releases/download/straight-curly-data/data.zip
!unzip data.zip

--2025-12-01 18:57:14--  https://github.com/SVizor42/ML_Zoomcamp/releases/download/straight-curly-data/data.zip
Resolving github.com (github.com)... 140.82.112.4
Connecting to github.com (github.com)|140.82.112.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/405934815/e712cf72-f851-44e0-9c05-e711624af985?sp=r&sv=2018-11-09&sr=b&spr=https&se=2025-12-01T19%3A52%3A58Z&rscd=attachment%3B+filename%3Ddata.zip&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2025-12-01T18%3A52%3A28Z&ske=2025-12-01T19%3A52%3A58Z&sks=b&skv=2018-11-09&sig=Wm00K2VGIQ2fc0ilRvn3MKBGEUU%2BrDrXRtB7ooXZzXk%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc2NDYxNzIzNCwibmJmIjoxNzY0NjE1NDM0LCJwYXRoIjoicmVsZWFzZWFzc2V0cHJvZHVjdGlvbi5i

In [30]:

import torch
import numpy as np
from PIL import Image

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [31]:
import torchvision.transforms as transforms

# Define the transformations
#
data_transforms = transforms.Compose([
    # 1. Resize the images to 200x200 pixels
    transforms.Resize((200, 200)),
    # 2. Convert the image (H, W, C) to a PyTorch Tensor (C, H, W) in [0, 1]
    transforms.ToTensor(),
    # 3. Normalize the tensor with mean and standard deviation for each channel
    # These are often standard values derived from large datasets like ImageNet,
    # or you can calculate them from your specific dataset.
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [46]:
import torchvision.datasets as datasets
import os

# Define the root directory where you unzipped the data
DATA_ROOT = 'data'

# Create the training and validation datasets
train_dir = os.path.join(DATA_ROOT, 'train')
test_dir = os.path.join(DATA_ROOT, 'test')

train_dataset = datasets.ImageFolder(
    root=train_dir,
    transform=data_transforms
)

validation_dataset = datasets.ImageFolder(
    root=test_dir,
    transform=data_transforms
)

# You can check the class-to-index mapping:
print(f"Class mapping: {train_dataset.class_to_idx}")

Class mapping: {'curly': 0, 'straight': 1}


In [47]:
from torch.utils.data import DataLoader

BATCH_SIZE = 64 # A common starting batch size

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,          # Shuffle for training data
    num_workers=4          # Use multiple processes for faster loading
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,         # No need to shuffle validation data
    num_workers=4
)

print(f"Total batches in training data: {len(train_loader)}")
print(f"Total images in training data: {len(train_dataset)}")

Total batches in training data: 13
Total images in training data: 800


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [34]:
import os
from torch.utils.data import Dataset

class Dataset(Dataset):
    def __init__(self, data_dir, transform=None):
        self.data_dir = data_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        self.classes = sorted(os.listdir(data_dir))
        self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}

        for label_name in self.classes:
            label_dir = os.path.join(data_dir, label_name)
            for img_name in os.listdir(label_dir):
                self.image_paths.append(os.path.join(label_dir, img_name))
                self.labels.append(self.class_to_idx[label_name])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

In [35]:
input_size = 224

# ImageNet normalization values
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

# Simple transforms - just resize and normalize
train_transforms = transforms.Compose([
    transforms.RandomRotation(10),           # Rotate up to 10 degrees
    transforms.RandomResizedCrop(224, scale=(0.9, 1.0)),  # Zoom
    transforms.RandomHorizontalFlip(),       # Horizontal flip
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

val_transforms = transforms.Compose([
    transforms.Resize((input_size, input_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

In [42]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

# Model definition
class BinaryCNN(nn.Module):
    def __init__(self):
        super(BinaryCNN, self).__init__()
        # Input shape: (3, 200, 200)

        # 1. Convolutional Layer (nn.Conv2d)
        # Input: (3, 200, 200)
        # Output: 32 filters, kernel=(3,3), padding=0, stride=1
        self.conv1 = nn.Conv2d(
            in_channels=3,
            out_channels=32,
            kernel_size=(3, 3),
            stride=1,
            padding=0
        )
        # Activation is applied in the forward pass (F.relu)

        # 2. Max Pooling (nn.MaxPool2d)
        # Pooling size: (2, 2)
        self.pool = nn.MaxPool2d(kernel_size=(2, 2))

        # --- Calculate the size of the feature map after conv and pool ---
        # 200 - 3 + 1 = 198 (size after convolution)
        # 198 / 2 = 99 (size after max pooling)
        # Number of features to flatten: 32 channels * 99 * 99 = 313632

        # 3. Flatten (nn.Flatten)
        # This is handled in the forward pass using nn.Flatten() or .view()

        # 4. First Linear Layer (nn.Linear)
        self.fc1 = nn.Linear(32 * 99 * 99, 64)

        # 5. Output Linear Layer (nn.Linear)
        # Output: 1 neuron for binary classification
        self.fc2 = nn.Linear(64, 1)


    def forward(self, x):
        # Apply Conv layer and ReLU
        x = F.relu(self.conv1(x))

        # Apply Max Pooling
        x = self.pool(x)

        # Flatten the feature maps into a vector
        # x.shape will be (batch_size, 32, 99, 99)
        # Start flattening from dimension 1 (leaving batch size intact)
        x = torch.flatten(x, 1)

        # First Linear Layer and ReLU
        x = F.relu(self.fc1(x))

        # Return the raw logit (output of the last linear layer)
        output = self.fc2(x)

        return output

# Instantiate the model
model = BinaryCNN()
print("Model structure defined:")
print(model)

Model structure defined:
BinaryCNN(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=(2, 2), stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=313632, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=1, bias=True)
)


In [43]:
def run_test_epoch(criterion_name, criterion, train_loader, device):
    """Initializes a new model and optimizer, trains for 1 epoch, and returns the final training loss."""
    print(f"\n--- Testing Criterion: {criterion_name} ---")

    # 1. Initialize a FRESH model for each test
    model = BinaryCNN().to(device)

    # 2. Define the requested optimizer
    optimizer = optim.SGD(model.parameters(), lr=0.002, momentum=0.8)

    model.train()
    total_loss = 0.0

    for i, data in enumerate(train_loader, 0):
        inputs, labels = data
        inputs = inputs.to(device)

        # *** CRITICAL: Loss-Specific Label Shaping ***

        if criterion_name == "nn.CrossEntropyLoss()":
            # CrossEntropyLoss expects long/int labels with shape (B)
            # We reshape the (B, 1) float labels back to (B) long/int
            labels = labels.squeeze().long().to(device)
        elif criterion_name == "nn.MSELoss()":
            # MSELoss expects float targets (B, 1)
            labels = labels.float().to(device)
        else:
            # BCEWithLogitsLoss and CosineEmbeddingLoss expect float targets (B, 1)
            labels = labels.float().unsqueeze(1).to(device)


        optimizer.zero_grad()

        # 3. Forward Pass
        outputs = model(inputs)

        # *** CRITICAL: Loss-Specific Output Shaping ***

        if criterion_name == "nn.CrossEntropyLoss()":
            # CrossEntropyLoss expects output to be (B, C), where C is the number of classes.
            # Our model output is (B, 1). We need to reshape the output to (B, 2)
            # to represent logit for class 0 and logit for class 1.
            # We'll use a placeholder for this test (often, you'd change the model
            # to have 2 output neurons for this loss), but for a proper test:
            # We'll stick to the 1-neuron output and use BCELoss for the final comparison.
            # We can't use CrossEntropyLoss directly with a single output neuron for 2 classes.

            # We will skip CrossEntropyLoss() as it requires a change to the model architecture (2 output neurons).
            print(f"SKIPPING {criterion_name}: Requires model output change (1 -> 2 neurons).")
            return float('inf')

        elif criterion_name == "nn.CosineEmbeddingLoss()":
            # CosineEmbeddingLoss expects three inputs: input1, input2, and target (1 or -1).
            # It's completely inappropriate for this task.
            print(f"SKIPPING {criterion_name}: Inappropriate for classification (requires two inputs).")
            return float('inf')

        elif criterion_name == "nn.MSELoss()":
            # MSELoss requires outputs to match label shape (B, 1)
            pass

        elif criterion_name == "nn.BCEWithLogitsLoss()":
            # BCEWithLogitsLoss requires outputs to match label shape (B, 1)
            pass

        # 4. Calculate Loss
        loss = criterion(outputs, labels)

        # 5. Backpropagate and Optimize
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"  Result: Avg Training Loss over 1 Epoch = {avg_loss:.4f}")
    return avg_loss

## --- 3. Run All Tests ---

results = {}

# Dictionary of criteria to test
criteria_to_test = {
    "nn.BCEWithLogitsLoss()": nn.BCEWithLogitsLoss(),
    "nn.MSELoss()": nn.MSELoss(),
    # nn.CrossEntropyLoss and nn.CosineEmbeddingLoss are incompatible
    # with the current model/task, but we leave the logic in run_test_epoch
    # to show why they fail.
    "nn.CrossEntropyLoss()": nn.CrossEntropyLoss(),
    "nn.CosineEmbeddingLoss()": nn.CosineEmbeddingLoss()
}


for name, criterion in criteria_to_test.items():
    loss = run_test_epoch(name, criterion, train_loader, device)
    if loss != float('inf'):
        results[name] = loss

# --- 4. Print Summary ---

print("\n" + "="*50)
print("FINAL LOSS FUNCTION COMPARISON (1-Epoch Training Loss)")
print("="*50)

# Sort results to clearly show the "best" (lowest loss)
sorted_results = sorted(results.items(), key=lambda item: item[1])

for name, loss in sorted_results:
    print(f"Loss: {loss:.4f} \t\t Criterion: {name}")

print("\nBased on theoretical suitability and this simple 1-epoch test:")
print(f"**{sorted_results[0][0]}** is the mathematically correct and best-performing loss function for your binary classification task.")


--- Testing Criterion: nn.BCEWithLogitsLoss() ---
  Result: Avg Training Loss over 1 Epoch = 0.6872

--- Testing Criterion: nn.MSELoss() ---


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/loss.py:634: UserWarning: Using a target size (torch.Size([64])) that is different to the input size (torch.Size([64, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/loss.py:634: UserWarning: Using a target size (torch.Size([32])) that is different to the input size (torch.Size([32, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


  Result: Avg Training Loss over 1 Epoch = 4.1244

--- Testing Criterion: nn.CrossEntropyLoss() ---
SKIPPING nn.CrossEntropyLoss(): Requires model output change (1 -> 2 neurons).

--- Testing Criterion: nn.CosineEmbeddingLoss() ---
SKIPPING nn.CosineEmbeddingLoss(): Inappropriate for classification (requires two inputs).

FINAL LOSS FUNCTION COMPARISON (1-Epoch Training Loss)
Loss: 0.6872 		 Criterion: nn.BCEWithLogitsLoss()
Loss: 4.1244 		 Criterion: nn.MSELoss()

Based on theoretical suitability and this simple 1-epoch test:
**nn.BCEWithLogitsLoss()** is the mathematically correct and best-performing loss function for your binary classification task.


In [44]:
# Option 1: Using torchsummary (install with: pip install torchsummary)
from torchsummary import summary
summary(model, input_size=(3, 200, 200))

# Option 2: Manual counting
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params}")

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 32, 198, 198]             896
         MaxPool2d-2           [-1, 32, 99, 99]               0
            Linear-3                   [-1, 64]      20,072,512
            Linear-4                    [-1, 1]              65
Total params: 20,073,473
Trainable params: 20,073,473
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.46
Forward/backward pass size (MB): 11.96
Params size (MB): 76.57
Estimated Total Size (MB): 89.00
----------------------------------------------------------------
Total parameters: 20073473


Question 2: 20073473

  
   
    ,,AnsAAA
  a

In [54]:
train_transforms = transforms.Compose([
    transforms.Resize((200, 200)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ), # ImageNet normalization
    transforms.RandomRotation(50),
    transforms.RandomResizedCrop(200, scale=(0.9, 1.0), ratio=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(),
  ])

In [55]:
# Get the device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
num_epochs = 10
history = {'acc': [], 'loss': [], 'val_acc': [], 'val_loss': []}

# Optimizer Setup
# Use torch.optim.SGD with specified parameters
optimizer = optim.SGD(
    model.parameters(),
    lr=0.002,
    momentum=0.8
)

# Define the correct criterion for binary classification
criterion = nn.BCEWithLogitsLoss()

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        labels = labels.float().unsqueeze(1) # Ensure labels are float and have shape (batch_size, 1)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        # For binary classification with BCEWithLogitsLoss, apply sigmoid to outputs before thresholding for accuracy
        predicted = (torch.sigmoid(outputs) > 0.5).float()
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = correct_train / total_train
    history['loss'].append(epoch_loss)
    history['acc'].append(epoch_acc)

    model.eval()
    val_running_loss = 0.0
    correct_val = 0
    total_val = 0
    with torch.no_grad():
        for images, labels in validation_loader:
            images, labels = images.to(device), labels.to(device)
            labels = labels.float().unsqueeze(1)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_running_loss += loss.item() * images.size(0)
            predicted = (torch.sigmoid(outputs) > 0.5).float()
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()

    val_epoch_loss = val_running_loss / len(validation_dataset)
    val_epoch_acc = correct_val / total_val
    history['val_loss'].append(val_epoch_loss)
    history['val_acc'].append(val_epoch_acc)

    print(f"Epoch {epoch+1}/{num_epochs}, "
          f"Loss: {epoch_loss:.4f}, Acc: {epoch_acc:.4f}, "
          f"Val Loss: {val_epoch_loss:.4f}, Val Acc: {val_epoch_acc:.4f}")

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Epoch 1/10, Loss: 0.2711, Acc: 0.8812, Val Loss: 0.7989, Val Acc: 0.6716
Epoch 2/10, Loss: 0.2232, Acc: 0.9150, Val Loss: 0.6310, Val Acc: 0.7264
Epoch 3/10, Loss: 0.2384, Acc: 0.9000, Val Loss: 0.7052, Val Acc: 0.6866
Epoch 4/10, Loss: 0.2264, Acc: 0.9038, Val Loss: 0.9145, Val Acc: 0.6567
Epoch 5/10, Loss: 0.1734, Acc: 0.9500, Val Loss: 0.8490, Val Acc: 0.6617
Epoch 6/10, Loss: 0.1466, Acc: 0.9513, Val Loss: 0.7696, Val Acc: 0.7015
Epoch 7/10, Loss: 0.0864, Acc: 0.9888, Val Loss: 0.7695, Val Acc: 0.7363
Epoch 8/10, Loss: 0.0766, Acc: 0.9875, Val Loss: 0.8037, Val Acc: 0.7214
Epoch 9/10, Loss: 0.0697, Acc: 0.9962, Val Loss: 0.7411, Val Acc: 0.7313
Epoch 10/10, Loss: 0.0554, Acc: 0.9950, Val Loss: 0.9965, Val Acc: 0.6965


Answer to question 3 is 0.84

Answer to question 4 is 0.078

Answer to 5 is 0.1567 is 0.08
  Answer to 6 is 0.7174 is is 0.68